# 🏅 Olympics Analytics Platform — Google Colab runner

Runs the **whole project** (data quality → statistics → SQL → forecasting → 14-model comparison → calibrated medal model → monitoring → report) and saves **everything to your Google Drive**.

**Disconnect-safe:** every stage and every model is checkpointed in Drive. If the runtime dies, just open the notebook and choose *Runtime → Run all* — finished stages are skipped and the run continues where it stopped.

**Before you start (one time)**
1. Put `olympics-analytics-platform.zip` in your Google Drive, in a folder called `Olympics-Analytics` (or upload it when cell 3 asks).
2. Create a free Kaggle API token (kaggle.com → Settings → API) and add it to Colab **Secrets** (🔑 icon on the left): `KAGGLE_USERNAME` and `KAGGLE_KEY` (or the new single `KAGGLE_API_TOKEN`). Open each dataset page once on kaggle.com and accept its terms.
3. No Kaggle yet? Tick `USE_DEMO_DATA` below to run everything on built-in synthetic data.

**Where things end up** (inside `MyDrive/Olympics-Analytics/`)

| Folder | Content |
|---|---|
| `project/` | the code |
| `workspace/data/` | raw download, cleaned data (parquet) |
| `workspace/reports/figures/` | all charts (PNG) + `FIGURES.md` captions |
| `workspace/reports/tables/` | every table as CSV (model comparison, tests, forecasts, drift…) |
| `workspace/reports/metrics/` | all metrics as JSON, `REPORT.html` is in `workspace/reports/` |
| `workspace/models/` | versioned models + model cards |
| `workspace/mlruns/`, `checkpoints/`, `feedback/`, `logs/` | experiment tracking, resume state, human feedback, logs |

## 1 · Settings

In [ ]:
#@title Settings { display-mode: "form" }
DRIVE_FOLDER   = "Olympics-Analytics"   #@param {type:"string"}
DATA_SOURCE    = "heesoo37_120y"         #@param ["heesoo37_120y", "olympedia"]
PROFILE        = "full"                  #@param ["fast", "full"]
USE_DEMO_DATA  = False                   #@param {type:"boolean"}
FORCE_STAGES   = ""                      #@param {type:"string"}
# FORCE_STAGES: space separated stages to recompute, e.g. "compare final"  (or "all"). Leave empty to resume.

## 2 · Mount Google Drive

In [ ]:
import os, sys, shutil, zipfile, subprocess, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
except ImportError:                       # running locally (for testing)
    IN_COLAB = False
    BASE = Path.home() / DRIVE_FOLDER

BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / "project"
ROOT = BASE / "workspace"                 # every output goes here
print("Drive folder :", BASE)
print("Outputs      :", ROOT)

## 3 · Get the code (first run only — kept in your Drive afterwards)

In [ ]:
def _find_zip():
    names = ["olympics-analytics-platform.zip"]
    for folder in (BASE, Path("/content"), Path("/content/drive/MyDrive"), Path.cwd()):
        for n in names:
            if (folder / n).exists():
                return folder / n
    return None

if not (PROJECT / "src" / "olympics").exists():
    z = _find_zip()
    if z is None and IN_COLAB:
        from google.colab import files
        print("Upload olympics-analytics-platform.zip ...")
        up = files.upload()
        z = Path("/content") / next(iter(up))
    assert z is not None, "Put olympics-analytics-platform.zip into the Drive folder and re-run this cell."
    tmp = Path("/tmp/_olympics_unzip"); shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(z) as zf: zf.extractall(tmp)
    inner = next(p for p in tmp.rglob("params.yaml")).parent
    PROJECT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(inner, PROJECT, dirs_exist_ok=True)
    print("Code installed to", PROJECT)
else:
    print("Code already in Drive:", PROJECT)

sys.path[:0] = [str(PROJECT / "src"), str(PROJECT)]
os.environ["PYTHONPATH"] = f"{PROJECT / 'src'}:{PROJECT}"
os.environ["OLYMPICS_ROOT"] = str(ROOT)
os.chdir(PROJECT)

## 4 · Install dependencies (≈2–3 min per fresh Colab session)

In [ ]:
need = {"statsmodels": "statsmodels", "xgboost": "xgboost", "lightgbm": "lightgbm", "catboost": "catboost", "shap": "shap",
        "mlflow": "mlflow", "evidently": "evidently", "kaggle": "kaggle", "pyarrow": "pyarrow", "tabulate": "tabulate",
        "markdown": "markdown", "gradio": "gradio", "fastapi": "fastapi", "uvicorn": "uvicorn", "plotly": "plotly", "httpx": "httpx"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing and IN_COLAB:
    print("Installing:", ", ".join(missing))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-800:] if r.returncode else "")
    importlib.invalidate_caches()
elif missing:
    print("Not installed (optional, skipped locally):", missing)
else:
    print("All dependencies present.")
import sklearn, pandas
print("pandas", pandas.__version__, "| scikit-learn", sklearn.__version__)

## 5 · Kaggle credentials

In [ ]:
if USE_DEMO_DATA:
    print("Demo mode: synthetic data, no Kaggle needed.")
else:
    def _secret(name):
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return os.environ.get(name)
    user, key, token = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY"), _secret("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    if user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
    have_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not (token or (user and key) or have_file):
        if IN_COLAB:
            from google.colab import files
            print("No Kaggle secret found. Upload your kaggle.json:")
            up = files.upload()
            (Path.home() / ".kaggle").mkdir(exist_ok=True)
            shutil.move(next(iter(up)), Path.home() / ".kaggle" / "kaggle.json")
            os.chmod(Path.home() / ".kaggle" / "kaggle.json", 0o600)
        else:
            print("No Kaggle credentials found (fine if data.local_dir is set).")
    print("Kaggle credentials ready.")

## 6 · Create the pipeline

In [ ]:
from olympics.config import load_config
from olympics.pipeline import Pipeline, STAGES

overrides = [f"data.source={DATA_SOURCE}", f"ml.profile={PROFILE}"]
cfg = load_config(overrides=overrides)
pipe = Pipeline(cfg, ROOT, demo=USE_DEMO_DATA)
FORCE = FORCE_STAGES.split()
print("Stages:", STAGES)
print("Already finished:", list(pipe.ck.status()) or "none")

def stage(*names, **kw):
    """Run stages (checkpoint-aware). Safe to re-run after a disconnect."""
    out = pipe.run(list(names), FORCE, **kw)
    return out

## 7 · Run the pipeline
Each cell can be run on its own and is safe to re-run.

### 7a · Download → validate → clean → SQL layer

In [ ]:
r = stage("ingest", "clean", "sql", force_download=False)
print(r)
from IPython.display import Markdown, display
display(Markdown((pipe.paths.tables / "data_quality_report.md").read_text()))

### 7b · Statistics, hypothesis tests and figures

In [ ]:
r = stage("stats"); print(r)
import pandas as pd
display(pd.read_csv(pipe.paths.tables / "medalist_vs_rest_tests.csv").round(3))

### 7c · Medal forecasting (rolling-origin backtest)

In [ ]:
r = stage("forecast"); print(r)
display(pd.read_csv(pipe.paths.tables / "forecast_model_comparison.csv"))
display(pd.read_csv(pipe.paths.tables / "forecast_next_edition.csv").head(10).round(1))

### 7d · Leakage-safe features → compare 14+ models → grouped CV → calibration → threshold → final model
⏱ The longest step (roughly 15–40 min on free Colab with `PROFILE=full`). **Each model is saved as soon as it finishes**, so a disconnect only loses the model that was running.

In [ ]:
r = stage("features", "compare", "final"); print(r)
display(pd.read_csv(pipe.paths.tables / "model_comparison_validation.csv")
        [["rank", "model", "pr_auc", "roc_auc", "f1_best_threshold", "brier", "ece", "fit_seconds"]].round(4))

### 7e · Monitoring + report

In [ ]:
r = stage("monitor", "report"); print(r)

## 8 · Results

In [ ]:
import json
fm = json.loads((pipe.paths.metrics / "final_model_metrics.json").read_text())
t, b = fm["test_metrics"], fm["baseline_test_metrics"]
print(f"Champion: {fm['algorithm']}  | version {fm['version']}  | calibration {fm['calibration']}  | threshold {fm['threshold']:.3f}\n")
rows = [(k, t[k], b.get(k)) for k in ["accuracy","balanced_accuracy","precision","recall","specificity","f1","f2","mcc","cohen_kappa","roc_auc","pr_auc","log_loss","brier","brier_skill_score","ece","mce","precision_at_top10pct","lift_at_top10pct"]]
display(pd.DataFrame(rows, columns=["metric", "test", "no-skill baseline"]).round(4))
print("95% cluster-bootstrap CIs:", {k: [round(x, 3) for x in v] for k, v in fm["test_confidence_intervals_95"].items()})
print("Temporal check (train on the past, test on the newest editions):", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in fm["temporal_backtest"].items() if k != "note"})
display(pd.read_csv(pipe.paths.tables / "cross_validation_top_models.csv").round(4))
display(pd.read_csv(pipe.paths.tables / "threshold_strategies.csv").round(3))

In [ ]:
# All figures (each has a "what it shows / why it matters" caption in FIGURES.md)
from IPython.display import Image, display
idx = json.loads((pipe.paths.figures / "figure_index.json").read_text())
for name, meta in sorted(idx.items()):
    print(f"\n{meta['title']}\n  what: {meta['what_it_shows']}\n  why : {meta['why_it_matters']}")
    display(Image(str(pipe.paths.figures / meta["file"]), width=760))

## 9 · Dashboard (Gradio) — filters, country comparison, drill-downs, predictor with explanations, feedback, SQL console

In [ ]:
from apps.dashboard.app import build_app
demo_app = build_app(str(ROOT))
demo_app.launch(share=True, debug=False)     # prints a public link valid while this session runs

## 10 · (Optional) FastAPI service inside Colab
Starts the REST API in the background and prints a link to the interactive docs (`/docs`).

In [ ]:
import threading, time, uvicorn
from apps.api.main import app as api_app
cfg_srv = uvicorn.Config(api_app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(cfg_srv)
threading.Thread(target=server.run, daemon=True).start(); time.sleep(3)
if IN_COLAB:
    from google.colab import output
    print("API docs:", output.eval_js("google.colab.kernel.proxyPort(8000)") + "docs")
import requests
print(requests.get("http://127.0.0.1:8000/health").json())
print(requests.post("http://127.0.0.1:8000/predict?explain=true", json={"Age": 24, "Height": 185, "Weight": 78, "Sex": "M", "NOC": "USA",
      "Sport": "Swimming", "Season": "Summer", "Year": 2016, "athlete_prev_medals": 3}).json())

## 11 · Human feedback → retraining
Predictions made in the dashboard/API are logged; confirm outcomes there (or in `feedback/feedback.jsonl`), then run this cell. The challenger replaces the champion only if it is not worse on the same untouched test split.

In [ ]:
from olympics.ml.retrain import retrain_with_feedback
from olympics.checkpoint import fingerprint
ml = pipe.ml_frame()
print(retrain_with_feedback(cfg, pipe.paths, ml, pipe.tracker, fingerprint(len(ml), cfg["data"]["source"], USE_DEMO_DATA), force=False))

## 12 · Package everything for download
Everything is already in Drive; this makes a single zip of the reports/models for convenience.

In [ ]:
out_zip = BASE / "olympics_results"
shutil.make_archive(str(out_zip), "zip", root_dir=ROOT, base_dir="reports")
print("Reports zip:", str(out_zip) + ".zip")
for name in ("models",):
    shutil.make_archive(str(BASE / f"olympics_{name}"), "zip", root_dir=ROOT, base_dir=name)
print("Models zip :", str(BASE / "olympics_models.zip"))
if IN_COLAB:
    from google.colab import files
    files.download(str(out_zip) + ".zip")

### Troubleshooting
* **Runtime disconnected** → reconnect, *Runtime → Run all*. Finished stages print `already complete - skipping`.
* **Kaggle 403 / not found** → accept the dataset's terms on its Kaggle page, then re-run cell 7a with `FORCE_STAGES = "ingest"`.
* **Switching to the `olympedia` dataset** → run `!python -m olympics.cli inspect-source --root $OLYMPICS_ROOT --set data.source=olympedia` and check the printed file/column names match the adapter (see README).
* **Out of memory** → set `PROFILE = "fast"` or add `ml.compare_max_rows=80000` to `overrides` in cell 6.